## Optional: Install Dependencies in Google Colab

Run this only if Colab reports a missing package such as `resampy`.

In [ ]:
# Uncomment if needed in Google Colab
# !pip install -q resampy


# RHD Classification from PCG Signals

This notebook develops a **Bidirectional LSTM (BiLSTM)** model for classifying phonocardiogram (PCG) recordings into **RHD** and **Healthy** classes. Audio recordings are converted into MFCC sequences, padded/truncated to a fixed length, and supplied to a recurrent neural network.

> **Project purpose:** This is a research/proof-of-concept implementation prepared to support an academic paper. It is not intended to be a clinical diagnostic system.

## Workflow

`REFERENCE.csv + PCG recordings → label preparation → MFCC extraction → fixed-length sequences → train/validation/test split → BiLSTM → validation threshold selection → final test evaluation`

## Dataset

The PCG dataset is **not included in this repository**. Follow the dataset placement instructions in the dataset-path section below.

## 1. Imports and Reproducibility

The notebook can be run in Google Colab or from a local clone of the repository. If using Colab, install the dependencies first if required.

In [ ]:
from pathlib import Path
import os
import random

import numpy as np
import pandas as pd
import librosa
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
)
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Bidirectional, Input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)

## 2. Dataset Paths

### Google Colab

Place the dataset in Google Drive as:

```text
My Drive/
└── training_rhd/
    ├── REFERENCE.csv
    ├── a0001.wav
    ├── a0002.wav
    └── ...
```

### Local GitHub clone

Place the dataset as `data/training_rhd/`. The dataset itself should not be committed if it is not redistributable.

The next cell automatically detects Google Colab and mounts Drive; otherwise it uses the repository's `data/training_rhd/` directory.

In [ ]:
# Detect Google Colab and set the dataset location
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')
    DATA_FOLDER = Path('/content/drive/MyDrive/training_rhd')
else:
    PROJECT_ROOT = Path.cwd()
    DATA_FOLDER = PROJECT_ROOT / 'data' / 'training_rhd'

CSV_PATH = DATA_FOLDER / 'REFERENCE.csv'
AUDIO_FOLDER = DATA_FOLDER

if not CSV_PATH.exists():
    raise FileNotFoundError(
        f'REFERENCE.csv not found at {CSV_PATH}. '
        'In Colab, place training_rhd in My Drive. '
        'Locally, place it under data/training_rhd/.'
    )

print('Running in Google Colab:', IN_COLAB)
print('Dataset folder:', DATA_FOLDER)
print('REFERENCE.csv:', CSV_PATH)
print('Dataset exists:', DATA_FOLDER.exists())
print('REFERENCE.csv exists:', CSV_PATH.exists())


## 3. Load and Prepare Labels

The original reference labels are converted from `{-1, 1}` into binary labels:

- `0` = **RHD**
- `1` = **Healthy**

This mapping is kept consistent throughout evaluation and threshold analysis.

In [ ]:
df = pd.read_csv(CSV_PATH)

required_columns = {'filename', 'label'}
missing_columns = required_columns - set(df.columns)
if missing_columns:
    raise ValueError(
        f'REFERENCE.csv is missing columns: {sorted(missing_columns)}'
    )

# Check that the dataset contains only the expected original labels.
# Original labels: -1 = RHD, 1 = Healthy
unique_labels = set(df['label'].dropna().unique())
expected_labels = {-1, 1}
if not unique_labels.issubset(expected_labels):
    raise ValueError(
        f'Unexpected labels found: {sorted(unique_labels)}. '
        f'Expected only {sorted(expected_labels)}.'
    )

# Convert labels to a consistent binary format.
# 0 = RHD
# 1 = Healthy
df['label'] = df['label'].map({-1: 0, 1: 1}).astype(int)
df['full_path'] = df['filename'].apply(
    lambda fname: AUDIO_FOLDER / f'{fname}.wav'
)

print('Total records:', len(df))
print('Class distribution:')
print(df['label'].map({0: 'RHD', 1: 'Healthy'}).value_counts())
df.head()


## 4. MFCC Feature Extraction

Each PCG recording is converted into **20 MFCC features**. Sequences are padded or truncated to **400 time steps** so that every sample has the same shape for the BiLSTM.

In [ ]:
N_MFCC = 20
MAX_PAD_LENGTH = 400

def extract_features(file_path, n_mfcc=N_MFCC, max_pad_len=MAX_PAD_LENGTH):
    """Load a PCG recording, extract MFCCs, and pad/truncate the time axis."""
    try:
        audio, sample_rate = librosa.load(file_path, res_type="kaiser_fast")
        mfccs = librosa.feature.mfcc(y=audio, sr=sample_rate, n_mfcc=n_mfcc)

        if mfccs.shape[1] > max_pad_len:
            mfccs = mfccs[:, :max_pad_len]
        else:
            pad_width = max_pad_len - mfccs.shape[1]
            mfccs = np.pad(
                mfccs,
                pad_width=((0, 0), (0, pad_width)),
                mode="constant",
            )
        return mfccs
    except Exception as exc:
        print(f"Error processing {file_path}: {exc}")
        return None

features = []
labels = []
failed_files = []

for _, row in df.iterrows():
    file_path = Path(row["full_path"])
    if not file_path.exists():
        failed_files.append(str(file_path))
        continue

    mfccs = extract_features(file_path)
    if mfccs is not None:
        features.append(mfccs)
        labels.append(row["label"])
    else:
        failed_files.append(str(file_path))

if not features:
    raise RuntimeError("No audio files were successfully processed.")

X = np.array(features, dtype=np.float32)
y = np.array(labels, dtype=np.int32)

# librosa output: (samples, MFCCs, time_steps)
# BiLSTM input: (samples, time_steps, MFCCs)
X = X.transpose(0, 2, 1)

print("Feature matrix shape:", X.shape)
print("Label vector shape:", y.shape)
print("Failed/missing files:", len(failed_files))

## 5. Train / Validation / Test Split

The original notebook used the test set as the validation set during training. That allows the test data to influence model selection through early stopping and learning-rate reduction.

Here the data is split into:

- **80% training**
- **10% validation**
- **10% test**

The test set is kept untouched until final evaluation.

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.20,
    random_state=SEED,
    stratify=y,
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=SEED,
    stratify=y_temp,
)

print(f"Training set:   {len(X_train)} samples")
print(f"Validation set: {len(X_val)} samples")
print(f"Test set:       {len(X_test)} samples")

print("\nTraining class distribution:", np.bincount(y_train))
print("Validation class distribution:", np.bincount(y_val))
print("Test class distribution:", np.bincount(y_test))

## 6. BiLSTM Model

The model uses two Bidirectional LSTM layers followed by dropout and dense layers for binary classification. The sigmoid output represents the model's estimated probability of the **Healthy (label 1)** class.

In [ ]:
input_shape = (X_train.shape[1], X_train.shape[2])

model = Sequential([
    Input(shape=input_shape),
    Bidirectional(LSTM(64, return_sequences=True)),
    Dropout(0.3),
    Bidirectional(LSTM(64)),
    Dropout(0.3),
    Dense(64, activation="relu"),
    Dropout(0.3),
    Dense(1, activation="sigmoid"),
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

model.summary()

## 7. Model Training

Early stopping and learning-rate reduction monitor **validation loss only**. The test set is not used during training.

In [ ]:
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True,
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.2,
    patience=5,
    min_lr=1e-5,
)

history = model.fit(
    X_train,
    y_train,
    epochs=100,
    batch_size=32,
    validation_data=(X_val, y_val),
    callbacks=[early_stopping, reduce_lr],
    verbose=1,
)

## 8. Training Curves

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(10, 5))
plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training and Validation Accuracy")
plt.legend()
plt.grid(True)
plt.show()

## 9. Threshold Selection on the Validation Set

Because the model's sigmoid output is the probability of **Healthy**, a lower probability indicates stronger evidence for **RHD**.

The decision threshold is selected using the **validation set**, maximizing F1-score for the RHD class. The test set remains untouched during this selection.

In [ ]:
y_val_prob = model.predict(X_val, verbose=0).ravel()

thresholds = np.arange(0.10, 0.90, 0.01)
precisions, recalls, f1_scores = [], [], []

for threshold in thresholds:
    # probability >= threshold -> Healthy (1); otherwise RHD (0)
    y_val_pred = (y_val_prob >= threshold).astype(int)
    precisions.append(precision_score(y_val, y_val_pred, pos_label=0, zero_division=0))
    recalls.append(recall_score(y_val, y_val_pred, pos_label=0, zero_division=0))
    f1_scores.append(f1_score(y_val, y_val_pred, pos_label=0, zero_division=0))

best_index = int(np.argmax(f1_scores))
best_threshold = float(thresholds[best_index])

print(f"Best validation threshold: {best_threshold:.2f}")
print(f"RHD precision: {precisions[best_index]:.3f}")
print(f"RHD recall:    {recalls[best_index]:.3f}")
print(f"RHD F1-score:  {f1_scores[best_index]:.3f}")

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(thresholds, precisions, label="Precision (RHD)")
plt.plot(thresholds, recalls, label="Recall (RHD)")
plt.plot(thresholds, f1_scores, label="F1-score (RHD)", linewidth=3)
plt.axvline(
    x=best_threshold,
    linestyle="--",
    label=f"Best threshold ({best_threshold:.2f})",
)
plt.title("Validation Metrics vs. Classification Threshold")
plt.xlabel("Healthy Probability Threshold")
plt.ylabel("Score")
plt.legend()
plt.grid(True)
plt.show()

## 10. Final Evaluation on the Test Set

The selected threshold is now applied **once to the held-out test set**. These results represent the final evaluation of the trained model.

In [ ]:
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
y_test_prob = model.predict(X_test, verbose=0).ravel()
y_test_pred = (y_test_prob >= best_threshold).astype(int)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_test_pred,
        labels=[0, 1],
        target_names=["RHD", "Healthy"],
        zero_division=0,
    )
)

In [ ]:
cm = confusion_matrix(y_test, y_test_pred, labels=[0, 1])

plt.figure(figsize=(7, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    xticklabels=["RHD", "Healthy"],
    yticklabels=["RHD", "Healthy"],
)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title(f"Test Confusion Matrix (threshold = {best_threshold:.2f})")
plt.show()

## 11. Save the Trained Model

The trained model can optionally be saved for later inference. The model artifact is not required for reproducing the experiment and may be excluded from version control if desired.

In [ ]:
MODEL_DIR = PROJECT_ROOT / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = MODEL_DIR / "rhd_bilstm.keras"
model.save(MODEL_PATH)
print(f"Model saved to: {MODEL_PATH}")

In [ ]:
PROJECT_ROOT = Path.cwd()
MODEL_DIR = PROJECT_ROOT / 'models'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = MODEL_DIR / 'rhd_bilstm.keras'
model.save(MODEL_PATH)
print(f'Model saved to: {MODEL_PATH}')


## 12. Notes

- This notebook uses the PCG recordings referenced by `REFERENCE.csv`.
- Label mapping: **0 = RHD**, **1 = Healthy**.
- The threshold is selected using validation data and then evaluated on the held-out test set.
- The reported experiment achieved approximately **76.69% test accuracy**, with **0.79 RHD F1-score** and **0.87 RHD recall** at the selected threshold in the current run.
- These results are presented as a research/proof-of-concept demonstration supporting an academic paper, not as evidence of clinical readiness.
- For clinical use, independent external validation, patient-level evaluation, calibration, and appropriate clinical studies would be required.